In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df=pd.read_excel(r"FIFA World Cup Dataset.csv.xlsx")
df

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['squad_total_market_value_eur'], df['fifa_points_pre_tournament'], alpha=0.6)
plt.xlabel('Squad Total Market Value (EUR)')
plt.ylabel('FIFA Points Pre-Tournament')
plt.title('Market Value vs FIFA Points')
plt.show()

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Load the 2026 Test Data
# (Make sure to use read_excel since the file is an Excel format despite the name)
df = pd.read_excel("FIFA World Cup Dataset.csv.xlsx", sheet_name="test")

# 2. Select Features
# These features mathematically dictate a team's strength and depth
features = ['fifa_points_pre_tournament', 'squad_total_market_value_eur', 'world_cup_titles_before']

# 3. Scale Features
# MinMaxScaler brings all values into a 0 to 1 range so Market Value doesn't overpower FIFA Points
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df[features])

# 4. Apply Weights for a Composite Score
# We weigh FIFA Points highest (50%), followed by Squad Value (40%), and historical pedigree (10%)
weights = [0.5, 0.4, 0.1]
composite_score = (scaled_features * weights).sum(axis=1)

# 5. Normalize into Probabilities
# We divide by the sum of all scores so the total probabilities across all 48 teams equals 100%
df['winning_probability_%'] = (composite_score / composite_score.sum()) * 100

# 6. Extract Team Names and their Winning Possibilities
result_df = df[['team', 'winning_probability_%']].sort_values(by='winning_probability_%', ascending=False).reset_index(drop=True)

# 7. Format and display the top results
result_df['winning_probability_%'] = result_df['winning_probability_%'].round(2).astype(str) + '%'
print("--- 2026 FIFA World Cup: Winning Possibilities ---")
print(result_df.to_string(index=True))

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# 1. Load the Data
# Using the exact CSV filename you uploaded
file_path = "FIFA World Cup Dataset.csv.xlsx"
print(f"Loading data from {file_path}...")
df = pd.read_excel(r"FIFA World Cup Dataset.csv.xlsx")
print("Data loaded successfully.\n")

# 2. Exploratory Data Analysis (EDA)
print("--- Data Shape ---")
print(f"Total Teams: {df.shape[0]}, Total Features: {df.shape[1]}\n")

# Define the features that dictate a team's strength
features = ['fifa_points_pre_tournament', 'squad_total_market_value_eur', 'world_cup_titles_before']

print("--- Sample of Features Used for Prediction ---")
print(df[['team'] + features].head())

# 3. Data Preprocessing & Feature Scaling
# We must scale the data because Market Value is in the billions and FIFA Points are in the thousands.
# MinMaxScaler brings all values into a 0.0 to 1.0 range so they can be accurately compared.
print("\nScaling features...")
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df[features])

# 4. Apply Heuristic Weights
# Assign importance to each feature. 
# 50% weight to current FIFA points, 40% to Squad Market Value, 10% to past World Cup titles
weights = [0.5, 0.4, 0.1]
composite_score = (scaled_features * weights).sum(axis=1)

# 5. Normalize into Winning Probabilities
# We divide each team's score by the sum of all scores.
# This ensures that the combined probability of all 48 teams equals exactly 100%.
df['winning_probability_%'] = (composite_score / composite_score.sum()) * 100

# 6. Formatting the Final Results
# Extract just the team name and probability, sort them from highest to lowest
result_df = df[['team', 'winning_probability_%']].sort_values(by='winning_probability_%', ascending=False).reset_index(drop=True)

# Create a clean string display column with the '%' sign and rounded to 2 decimal places
result_df['winning_probability_display'] = result_df['winning_probability_%'].round(2).astype(str) + '%'

print("\n==================================================")
print("   2026 FIFA WORLD CUP: WINNING POSSIBILITIES")
print("==================================================")
# Print the full ranked list to the console
print(result_df[['team', 'winning_probability_display']].to_string(index=True))

# 7. Visualization
# Plot a bar chart of the Top 10 most likely winners
plt.figure(figsize=(10, 6))
top_10 = result_df.head(10)

# Create the bar chart
bars = plt.bar(top_10['team'], top_10['winning_probability_%'], color='royalblue', edgecolor='black')

# Add the percentage text on top of each bar for readability
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f'{yval:.2f}%', ha='center', va='bottom', fontweight='bold')

plt.title('Top 10 Teams - 2026 FIFA World Cup Winning Probability', fontsize=14, fontweight='bold')
plt.ylabel('Winning Probability (%)', fontsize=12)
plt.xlabel('National Team', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, max(top_10['winning_probability_%']) + 1) # Add some headroom for the labels
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Display the plot
plt.show()